# Web Crawling for RAG — Async Fetching + Content-Extraction Library Showdown

Both `l23_web_crawling_for_rag (1).ipynb` and `l23_web_crawling_manual_approach.ipynb` focused on
*discovery* — how a crawler finds and fetches pages. This notebook assumes fetching is solved and
asks the other half of the question: once you have raw HTML, which library turns it into the
cleanest text for a RAG pipeline?

Three extraction methods, run on the same fetched pages:

| Method | Library | Approach |
|---|---|---|
| **Raw text** | `BeautifulSoup.get_text()` | Everything inside `<body>`, no filtering |
| **Readability** | `readability-lxml` | Arc90's "reader mode" algorithm — the same heuristic browsers use to strip clutter |
| **Trafilatura** | `trafilatura` | Purpose-built for NLP/RAG corpora — extracts main content + metadata (title, author, date) |

Fetching itself also gets an upgrade here: instead of the sequential `requests` loop from the
manual-crawl notebook, pages are fetched **concurrently** with `aiohttp`, rate-limited with an
`asyncio.Semaphore` rather than a blocking `time.sleep`.

**Target site:** `https://quotes.toscrape.com` — a second sandbox site from the same
scraping-practice family as `books.toscrape.com`, used here instead so the fetched pages (and
therefore the extraction comparison) aren't identical to the manual-crawl notebook.

**Ground rules:** same-domain only, a handful of pages (page budget below), a descriptive
`User-Agent`, and a concurrency cap low enough that it's indistinguishable from a single polite
visitor browsing quickly — not a load test.


In [ ]:
# pip install aiohttp beautifulsoup4 trafilatura readability-lxml   (already added via `uv add`)

import asyncio
from urllib.parse import urljoin, urlparse

import aiohttp
from bs4 import BeautifulSoup
from langchain_core.documents import Document

BASE_URL = "https://quotes.toscrape.com/"
ALLOWED_HOST = "quotes.toscrape.com"
PAGE_PATHS = [f"/page/{i}/" for i in range(1, 6)]   # 5 pages, known pagination pattern
MAX_CONCURRENT = 3       # politeness cap, not a performance target
USER_AGENT = "genai-class-educational-crawler/1.0"

def same_domain(url: str) -> bool:
    return urlparse(url).netloc.lower() == ALLOWED_HOST


## 1. Concurrent fetching with a semaphore

`asyncio.Semaphore(MAX_CONCURRENT)` caps how many requests are in flight at once — the async
equivalent of Scrapy's `CONCURRENT_REQUESTS` setting, and a more scalable version of the
sequential `time.sleep` delay used in the manual-crawl notebook.


In [ ]:
async def fetch(session: aiohttp.ClientSession, path: str, sem: asyncio.Semaphore) -> tuple[str, str]:
    url = urljoin(BASE_URL, path)
    async with sem:
        async with session.get(url, headers={"User-Agent": USER_AGENT}, timeout=10) as resp:
            html = await resp.text()
            return url, html

async def fetch_all(paths: list[str]) -> dict[str, str]:
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[fetch(session, p, sem) for p in paths])
    return {url: html for url, html in results if same_domain(url)}

pages = await fetch_all(PAGE_PATHS)
print(f"{len(pages)} pages fetched concurrently")


## 2. Extraction method 1 — raw `BeautifulSoup.get_text()`

The baseline every other method is measured against: all text in `<body>`, pagination controls
and sidebar tags included.


In [ ]:
sample_url, sample_html = next(iter(pages.items()))

raw_text = BeautifulSoup(sample_html, "lxml").get_text(separator=" ", strip=True)
print(f"raw get_text(): {len(raw_text):,} chars")
print(raw_text[:300])


## 3. Extraction method 2 — `readability-lxml`

Readability's algorithm scores each block of HTML by text-to-markup density and keeps only the
highest-scoring region — the same heuristic behind Firefox's and Safari's "reader view."


In [ ]:
from readability import Document as ReadabilityDocument

readability_doc = ReadabilityDocument(sample_html)
readability_title = readability_doc.short_title()
readability_html = readability_doc.summary()
readability_text = BeautifulSoup(readability_html, "lxml").get_text(separator=" ", strip=True)

print(f"readability title: {readability_title}")
print(f"readability text:  {len(readability_text):,} chars")
print(readability_text[:300])


## 4. Extraction method 3 — `trafilatura`

Trafilatura is built specifically for turning crawled web pages into clean NLP/RAG corpora at
scale — it extracts the main content and, where available, structured metadata (title, author,
date) in one call.


In [ ]:
import trafilatura

trafilatura_text = trafilatura.extract(sample_html, include_comments=False) or ""
trafilatura_metadata = trafilatura.extract_metadata(sample_html)

print(f"trafilatura text:   {len(trafilatura_text):,} chars")
print(f"trafilatura title:  {trafilatura_metadata.title if trafilatura_metadata else None}")
print(trafilatura_text[:300])


## 5. Side-by-side comparison across all fetched pages

Char count alone doesn't prove quality — a method that strips too aggressively can lose real
content, not just noise. Printing all three side by side per page makes it easy to eyeball which
one actually kept the quotes and dropped the pagination links.


In [ ]:
comparison_rows = []
for url, html in pages.items():
    raw_len = len(BeautifulSoup(html, "lxml").get_text(separator=" ", strip=True))
    read_len = len(
        BeautifulSoup(ReadabilityDocument(html).summary(), "lxml").get_text(separator=" ", strip=True)
    )
    traf_len = len(trafilatura.extract(html, include_comments=False) or "")
    comparison_rows.append((url, raw_len, read_len, traf_len))

print(f"{'URL':<45}{'raw':>8}{'readability':>14}{'trafilatura':>14}")
for url, raw_len, read_len, traf_len in comparison_rows:
    print(f"{url:<45}{raw_len:>8}{read_len:>14}{traf_len:>14}")


## 6. Building `Document`s with the chosen extractor

`trafilatura` tends to win on a content-heavy page like this one — it tracks closest to "just the
quotes," with the extraction method recorded in metadata so it's traceable later if a different
page type needs a different extractor.


In [ ]:
documents = [
    Document(
        page_content=trafilatura.extract(html, include_comments=False) or "",
        metadata={"source": url, "tool": "trafilatura"},
    )
    for url, html in pages.items()
]

print(len(documents), "Documents ready for chunking")
print(documents[0].page_content[:300])


## Summary

| Method | Library | Strips nav/boilerplate? | Metadata extraction | Best for |
|---|---|---|---|---|
| **Raw text** | `BeautifulSoup` | No | None | Debugging, or pages that are already pure content |
| **Readability** | `readability-lxml` | Yes (density heuristic) | Title only | Article-style pages (blogs, news) |
| **Trafilatura** | `trafilatura` | Yes (purpose-built for corpora) | Title, author, date | RAG/NLP pipelines ingesting many heterogeneous pages |

## Takeaways

- Fetching and extraction are two separate problems — `l23_web_crawling_for_rag (1).ipynb` and
  the manual-crawl notebook both focused on fetching; this one isolates extraction and shows it
  matters just as much for RAG quality.
- `asyncio` plus a semaphore scales further than a sequential `time.sleep` loop while staying just
  as polite — the concurrency cap, not the absence of delay, is what keeps it respectful.
- Raw `get_text()` is a reasonable baseline but keeps nav/boilerplate; `readability-lxml` and
  `trafilatura` both solve that, with `trafilatura` additionally pulling structured metadata for
  free.
- As in the other crawling notebooks, every method here ends at the same place — a list of
  `Document` objects — so the choice of extractor is an input-quality decision, not something
  downstream chunking/embedding code needs to know about.
